## UAV Intrusion Detection Stacked Ensemble
This notebook uses One-Class SVM, Autoencoder, and Local Outlier Factor as base models, and combines their outputs with a meta-classifier ensemble.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score


### Load the Dataset

### Preprocessing

In [ ]:
from gps_ids import protocol, training

EXP = protocol.ensemble_heldout("2d")
r = training.run("StackedEnsemble_RF", EXP)
stacked_pred = r.y_pred
y_meta_test = EXP.y_test


### Train Base Models

### Meta-Classifier Ensemble

In [7]:
# Evaluation
report = classification_report(
    y_meta_test,
    stacked_pred,
    labels=[0, 1, 2, 3],
    target_names=list(EXP.class_names),
    zero_division=0,
    output_dict=True
)
f1 = report["weighted avg"]["f1-score"]
accuracy = accuracy_score(y_meta_test, stacked_pred)
print(classification_report(
    y_meta_test,
    stacked_pred,
    labels=[0, 1, 2, 3],
    target_names=list(EXP.class_names),
    zero_division=0
))
print(f"F1 Score: {f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")

              precision    recall  f1-score   support

      benign       0.99      1.00      0.99    397825
     attack1       0.98      0.95      0.97     36458
     attack2       1.00      0.99      1.00     44232
     attack3       0.99      0.95      0.97     32015

    accuracy                           0.99    510530
   macro avg       0.99      0.97      0.98    510530
weighted avg       0.99      0.99      0.99    510530

F1 Score: 0.9907
Accuracy: 0.9908


In [ ]:
# Seed sweep: the remaining seeds in training.SEEDS, appended to the results CSV.
SWEEP_SEEDS = [s for s in training.SEEDS if s != protocol.SEED]

training.run_seeds("StackedEnsemble_RF", lambda s: protocol.ensemble_heldout("2d", seed=s), seeds=SWEEP_SEEDS)
